# J3 Après-midi : Analyse Multimodale (OCR, Audio, Vision) avec les LLMs

Bienvenue ! Cet après-midi, nous allons explorer comment utiliser les Modèles de Langage (LLMs) pour traiter des données "non textuelles" fréquentes en sciences sociales :
1. **OCR & PDF** : Extraire et nettoyer le texte d'un document numérisé imparfait.
2. **Audio & Vidéo** : Récupérer les sous-titres d'un discours politique sur YouTube et les classifier.
3. **Vision** : Analyser l'évolution de l'image politique à travers le temps (ex: photos de campagne vs exercice du pouvoir).

Nous utiliserons l'API OpenAI, les techniques de *Prompt Engineering* vues à la session précédente (Zero-shot, Few-shot, Chain of Thought), et des bibliothèques Python spécifiques.

## 0. Installation et configuration de l'environnement

*(Note : l'installation de `tesseract-ocr` prend quelques secondes. Elle nous permet de faire de l'OCR en français.)*

In [ ]:
!apt-get update -qq && apt-get install -y tesseract-ocr tesseract-ocr-fra > /dev/null
!pip install pytesseract pdf2image youtube-transcript-api openai pandas pillow requests

In [ ]:
import os
import requests
import pandas as pd
from PIL import Image
from io import BytesIO
from openai import OpenAI

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except ImportError:
    api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("ATTENTION : Aucune clé API trouvée. Veuillez configurer OPENAI_API_KEY dans les secrets Colab.")
else:
    client = OpenAI(api_key=api_key)
    print("Client OpenAI initialisé avec succès !")

## 1. OCR et correction par LLM : Lire les archives

Souvent en recherche (histoire politique, sociologie), on fait face à des scans de journaux ou des documents administratifs anciens. Tesseract est l'outil open-source standard pour l'OCR, mais il fait des erreurs, notamment sur les mises en page complexes (colonnes, petits caractères).
Nous allons télécharger la une du journal "L'Aurore" (le célèbre "J'accuse...!" de Zola) et voir comment un LLM peut corriger les erreurs de l'OCR.

In [ ]:
import pytesseract

# On télécharge l'image de la une de L'Aurore depuis Wikimedia
url_jaccuse = "https://upload.wikimedia.org/wikipedia/commons/4/4e/J%27accuse_1.jpg"
response = requests.get(url_jaccuse)
img_jaccuse = Image.open(BytesIO(response.content))

# Afficher l'image (redimensionnée pour un affichage rapide)
display(img_jaccuse.resize((400, 600)))

# On applique l'OCR brut (langue française)
texte_brut = pytesseract.image_to_string(img_jaccuse, lang='fra')

print("--- Extrait de l'OCR brut ---")
print(texte_brut[:500]) # On affiche les 500 premiers caractères

L'OCR brut contient des erreurs de reconnaissance et mélange souvent les colonnes. 
Utilisons `gpt-4o-mini` pour "nettoyer" le texte et le structurer.

In [ ]:
prompt_ocr = """
Tu es un historien expert.
Voici le résultat brut de l'OCR de la une d'un journal historique ("J'accuse" de Zola).
Ton travail est de :
1. Corriger les coquilles et les erreurs de l'OCR.
2. Restructurer le texte pour qu'il soit lisible (titre, sous-titres, début du corps du texte).
3. Produire un court résumé (2 phrases) du thème principal.

Texte OCR brut :
{texte}
"""

if api_key:
    try:
        reponse = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Tu es un expert en traitement de texte d'archives."},
                {"role": "user", "content": prompt_ocr.format(texte=texte_brut[:1500])} # On limite la taille pour l'exemple
            ],
            temperature=0.1
        )
        texte_propre = reponse.choices[0].message.content
        print("--- Texte corrigé par le LLM ---")
        print(texte_propre)
    except Exception as e:
        print(f"Erreur API : {e}")

### Hack-Time 🛠️

Modifiez le prompt ci-dessus pour demander au LLM d'extraire spécifiquement le nom des personnalités mentionnées dans l'article, formaté sous forme de liste à puces.

In [ ]:
# À vous de jouer : copiez le code ci-dessus, modifiez prompt_ocr et exécutez la requête.



## 2. Analyse de discours politique : Transcription YouTube et Classification

Plutôt que d'utiliser des fichiers audio lourds et de devoir configurer `Whisper` localement, nous pouvons extraire directement les sous-titres générés par YouTube d'une vidéo politique.
Nous allons ensuite utiliser les techniques de *Prompt Engineering* de la session J2 (*Few-Shot* et *Chain of Thought*) pour analyser l'idéologie du discours.

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi

# ID de la vidéo YouTube (Discours sur l'Europe à la Sorbonne d'Emmanuel Macron - 25 Avril 2024)
# URL: https://www.youtube.com/watch?v=0hLzL0yXJ9E
video_id = "0hLzL0yXJ9E"

try:
    # On récupère la transcription en français
    transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=['fr'])
    
    # On regroupe les 40 premières phrases pour avoir un bon extrait
    segments = [t['text'] for t in transcript[10:50]] # On saute l'intro
    texte_discours = " ".join(segments)
    
    print("--- Extrait du discours récupéré ---")
    print(texte_discours)
except Exception as e:
    print(f"Erreur lors de la récupération : {e}")

Appliquons la technique **Chain of Thought (CoT)** combinée au **Few-Shot** pour analyser la rhétorique de ce discours et classer la dimension idéologique prédominante.

In [ ]:
cot_prompt = """
Tu es un politologue expert en analyse de discours.
Analyse l'extrait de discours suivant.

Détermine la dimension idéologique principale abordée parmi :
- SOUVERAINETÉ ET PUISSANCE
- LIBÉRALISME ÉCONOMIQUE
- ÉCOLOGIE ET TRANSITION
- AUTRE

Procède en deux étapes (Chain of Thought) :
1. Raisonnement : Explique brièvement quels mots-clés ou concepts justifient ton choix.
2. Label : Termine par "LABEL: [Ta_Catégorie]".

Extrait :
"{extrait}"
"""

if api_key:
    try:
        reponse = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": cot_prompt.format(extrait=texte_discours)}],
            temperature=0.2 # On veut une analyse stable
        )
        print("--- Analyse du LLM ---")
        print(reponse.choices[0].message.content)
    except Exception as e:
        print(f"Erreur API : {e}")

### Hack-Time 🛠️

Trouvez la vidéo d'un autre responsable politique sur YouTube (copiez l'ID, qui est la suite de caractères après `v=`). 
Modifiez le prompt (ajoutez par exemple un exemple *Few-Shot*) pour que le modèle extrait les **arguments pour** et les **arguments contre** une mesure spécifique.

In [ ]:
# À vous de jouer :
# mon_video_id = "..."
# ...



## 3. Vision : Analyser l'évolution de l'image politique

Les Modèles Multimodaux comme `gpt-4o` peuvent "voir" les images. C'est extrêmement utile pour coder des symboles politiques dans des photographies de presse, affiches ou sur les réseaux sociaux (Instagram, Bluesky).

Comparons deux images montrant l'évolution d'Emmanuel Macron dans le temps (un portrait de campagne vs un portrait en exercice).

In [ ]:
# Image 1: Emmanuel Macron, portrait de campagne ou début de mandat (style ouvert)
url_img1 = "https://upload.wikimedia.org/wikipedia/commons/thumb/f/f4/Emmanuel_Macron_in_2017.jpg/400px-Emmanuel_Macron_in_2017.jpg"

# Image 2: Emmanuel Macron, portrait de 2023 (posture d'exercice du pouvoir / crise)
url_img2 = "https://upload.wikimedia.org/wikipedia/commons/thumb/f/f3/Emmanuel_Macron_2023_%28cropped%29.jpg/400px-Emmanuel_Macron_2023_%28cropped%29.jpg"

print("Image 1 (2017) :")
display(Image.open(BytesIO(requests.get(url_img1).content)))

print("Image 2 (2023) :")
display(Image.open(BytesIO(requests.get(url_img2).content)))

Demandons au modèle Vision de coder de manière systématique des variables d'intérêt pour comparer la mise en scène du pouvoir.

In [ ]:
prompt_vision = """
Tu es un chercheur en communication politique.
Analyse cette image d'un responsable politique. Code les variables suivantes au format JSON valide :
{
  "posture": "description de la posture physique",
  "regard": "face / fuyant / en l'air / autre",
  "arriere_plan": "description du décor",
  "symboles_visibles": ["symbole1", "symbole2"],
  "impression_degagee": "analyse en une phrase du message visuel"
}
Réponds UNIQUEMENT avec le bloc JSON sans aucun autre texte.
"""

def analyser_image(url):
    if not api_key: return "Clé API manquante"
    try:
        reponse = client.chat.completions.create(
            model="gpt-4o", # On utilise le modèle multimodale complet
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt_vision},
                        {"type": "image_url", "image_url": {"url": url}}
                    ]
                }
            ],
            max_tokens=300,
            temperature=0.0 # On veut de la consistance dans le codage JSON
        )
        return reponse.choices[0].message.content
    except Exception as e:
        return f"Erreur : {e}"

print("--- Analyse Image 1 ---")
print(analyser_image(url_img1))

print("\n--- Analyse Image 2 ---")
print(analyser_image(url_img2))

### Hack-Time 🛠️

Testez l'analyse sur une affiche de campagne historique (ex: Mitterrand "La Force Tranquille") ou une image récente issue d'un compte Bluesky/Instagram de politique.
Remplacez l'URL et ajustez le prompt pour détecter la présence de texte (slogan) ou de figures stéréotypées.

In [ ]:
# À vous de jouer :
# url_affiche = "..."
# ...


## 4. Discussion méthodologique

L'utilisation de LLMs multimodaux modifie profondément les méthodologies qualitatives et quantitatives :

* **Forces** : Gain de temps considérable, automatisation de l'annotation d'archives (OCR + correction) et possibilité de croiser l'analyse de texte, d'audio (YouTube) et d'images à grande échelle.
* **Limites** : 
  * **Hallucinations** : Le modèle peut "inventer" un drapeau en arrière-plan ou halluciner un texte sur une affiche floue.
  * **Biais idéologiques/culturels** : L'interprétation des postures (ex: ce qu'est une "posture présidentielle") dépend des normes culturelles d'entraînement du modèle.
  * **Manque de transparence** : Le modèle `gpt-4o` est une boîte noire.
* **Questions éthiques** : Analyser massivement des visages ou des discours soulève des questions de respect de la vie privée et de droit d'auteur. 

**L'automatisation ne remplace pas l'interprétation critique du chercheur en sciences sociales.**